# Semantic Similarity with TF-IDF and Cosine Similarity

**Tools:** Python, scikit-learn, NumPy, matplotlib

This notebook converts 10 sentences into numerical TF-IDF vectors, calculates pairwise cosine similarity, builds a reusable `find_similar(query, corpus, top_k)` function, tests related/unrelated queries, and visualises the 10×10 similarity matrix.

The notebook intentionally explains each new command as it is introduced.

## 1. Why do we need numerical vectors?

A computer can compare strings exactly, but raw text is not naturally a mathematical object. For example, `dog` and `puppy` are different strings even though people may recognise them as related concepts.

Machine-learning algorithms work with numbers. A vector representation turns each sentence into a point in a high-dimensional feature space. Once sentences are vectors, we can use mathematical operations such as dot products, vector norms, angles, and similarity scores.

In this project, **TF-IDF** creates the vectors and **cosine similarity** compares their directions.

**Important limitation:** TF-IDF is a lexical/statistical method, not a true language-understanding model. It mainly rewards shared vocabulary and contextual words. Modern embeddings are better at capturing synonyms such as `dog` and `puppy` even when they do not share words.

## 2. Import the libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### New commands explained

- `import numpy as np` imports NumPy and gives it the short name `np`; NumPy handles numerical arrays and matrices.
- `import matplotlib.pyplot as plt` imports matplotlib's plotting interface.
- `TfidfVectorizer` converts text documents into TF-IDF feature vectors.
- `cosine_similarity` calculates cosine similarity between vectors.
- `as np` and `as plt` are aliases that let us use shorter names.

## 3. Create the 10-sentence corpus

The corpus has three topics:

- **Pets/animals:** sentences 1–4
- **Travel:** sentences 5–7
- **Technology:** sentences 8–10

The sentences contain overlapping vocabulary inside each topic so that the TF-IDF similarity pattern can be observed.

In [ ]:
corpus = [
    "The dog is a friendly pet that enjoys daily walks.",
    "A puppy is a friendly animal that loves playing with its owner.",
    "Dogs make loyal pets and enjoy outdoor walks with people.",
    "Cats are popular pets that enjoy quiet homes and gentle care.",
    "Travelers enjoy visiting beautiful cities during summer holidays.",
    "A train journey can take tourists through beautiful countryside and cities.",
    "Tourists often book hotels before traveling to popular destinations.",
    "Modern smartphones use powerful processors and long-lasting batteries.",
    "A laptop needs a fast processor and enough memory for software development.",
    "New computer software can improve productivity and protect digital data."
]

labels = [
    "Pet 1", "Pet 2", "Pet 3", "Pet 4",
    "Travel 1", "Travel 2", "Travel 3",
    "Tech 1", "Tech 2", "Tech 3"
]

for index, sentence in enumerate(corpus):
    print(f"{index}: {sentence}")

### New commands explained

- A Python list stores the 10 sentences.
- `labels` stores short names for the heatmap axes.
- `enumerate(corpus)` gives both the position and the sentence.
- `f"{index}: {sentence}"` is an f-string that inserts variable values into the printed text.

## 4. Convert text into TF-IDF vectors

**TF-IDF = Term Frequency × Inverse Document Frequency.**

A term is useful when it is important in a particular sentence but is not equally common across the whole corpus. `TfidfVectorizer` learns the vocabulary and assigns each sentence a numerical vector.

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

### What happens here?

- `TfidfVectorizer()` creates the vectoriser.
- `fit_transform(corpus)` first **learns** the vocabulary (`fit`) and then **converts** the sentences into TF-IDF vectors (`transform`).
- `tfidf_matrix` contains 10 rows because there are 10 sentences.
- The number of columns equals the number of TF-IDF features learned from the corpus.

## 5. Inspect the vocabulary and vectors

In [ ]:
feature_names = vectorizer.get_feature_names_out()

print("Number of features:", len(feature_names))
print("First 30 features:")
print(feature_names[:30])

dense_matrix = tfidf_matrix.toarray()
print("\nDense matrix shape:", dense_matrix.shape)
print("\nTF-IDF vector for sentence 1:")
print(dense_matrix[0])

### New commands explained

- `get_feature_names_out()` returns the learned vocabulary.
- `toarray()` changes scikit-learn's memory-efficient sparse matrix into a normal NumPy array for easier inspection.
- `dense_matrix[0]` selects the first sentence's vector.

At this point, the important transformation is:

**sentence → words/features → numbers**

Those numbers allow mathematical comparison.

## 6. Calculate the 10×10 cosine-similarity matrix

Cosine similarity is:

$$
cos(θ) = \frac{A · B}{||A|| ||B||}
$$

`A · B` is the dot product and `||A||`/`||B||` are vector lengths.

Cosine similarity compares the **angle/direction** of vectors rather than their raw lengths. For non-negative TF-IDF vectors, scores are normally between 0 and 1.

In [ ]:
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)
print("\n10×10 similarity matrix:")
print(np.round(similarity_matrix, 3))

### New commands explained

- `cosine_similarity(tfidf_matrix)` compares every sentence with every other sentence.
- The result is a 10×10 matrix because there are 10 sentences.
- `np.round(..., 3)` rounds the displayed values to three decimal places.

The diagonal should be approximately `1.0`, because every sentence is perfectly similar to itself.

## 7. Manually verify the cosine formula

Understanding the library function is important, so we calculate one score ourselves.

In [ ]:
vector_a = dense_matrix[0]
vector_b = dense_matrix[1]

dot_product = np.dot(vector_a, vector_b)
norm_a = np.linalg.norm(vector_a)
norm_b = np.linalg.norm(vector_b)

manual_cosine = dot_product / (norm_a * norm_b)

print("Dot product:", dot_product)
print("Norm A:", norm_a)
print("Norm B:", norm_b)
print("Manual cosine:", manual_cosine)
print("scikit-learn cosine:", similarity_matrix[0, 1])

### New NumPy commands explained

- `np.dot(A, B)` calculates the dot product.
- `np.linalg.norm(A)` calculates the Euclidean length of a vector.
- The division directly implements the cosine formula.

The manual value should match scikit-learn's value, apart from tiny floating-point differences.

## 8. Build `find_similar(query, corpus, top_k)`

The function accepts a new sentence, the corpus, and the number of results requested.

A crucial point is that the query must be transformed using the **same fitted vectorizer** as the corpus. Otherwise the query and corpus vectors would not share the same feature dimensions.

In [ ]:
def find_similar(query, corpus, top_k=3):
    """Return the top_k corpus sentences most similar to a query."""

    query_vector = vectorizer.transform([query])
    corpus_vectors = vectorizer.transform(corpus)
    scores = cosine_similarity(query_vector, corpus_vectors)[0]

    ranked_indices = np.argsort(scores)[::-1]
    top_indices = ranked_indices[:top_k]

    results = []
    for index in top_indices:
        results.append((corpus[index], scores[index]))

    return results

### Function explained line by line

- `def find_similar(...)` creates a reusable function.
- `top_k=3` gives a default of three results.
- `vectorizer.transform([query])` converts the new query into the existing TF-IDF feature space.
- `vectorizer.transform(corpus)` creates the corpus vectors using the same vocabulary.
- `cosine_similarity(...)` calculates the query's score against every corpus sentence.
- `np.argsort(scores)` gives indices ordered by score.
- `[::-1]` reverses that order so the highest score comes first.
- `[:top_k]` keeps only the requested number of matches.
- The function returns `(sentence, score)` pairs.

## 9. Test 1 — related dog/puppy query

In [ ]:
query = "A puppy is a friendly pet that enjoys walks."

print("Query:", query)
print("\nTop matches:")

for rank, (sentence, score) in enumerate(find_similar(query, corpus, top_k=5), start=1):
    print(f"{rank}. Score = {score:.3f} | {sentence}")

### Interpretation

The query should rank pet-related sentences highly because they share words such as `puppy`, `friendly`, `pet`, `enjoys`, and `walks`.

Notice the important limitation: TF-IDF does **not** know from world knowledge that a puppy is a young dog. The similarity comes from the words and context represented in the corpus.

## 10. Test 2 — unrelated dog/airplane query

In [ ]:
query = "The airplane carried passengers to a distant city."

print("Query:", query)
print("\nTop matches:")

for rank, (sentence, score) in enumerate(find_similar(query, corpus, top_k=5), start=1):
    print(f"{rank}. Score = {score:.3f} | {sentence}")

The airplane query should favour travel-related sentences because of shared terms such as `city` and travel-related vocabulary. Pet sentences should generally receive lower scores.

This demonstrates why both **high-similarity** and **low-similarity** tests are necessary.

## 11. Test several queries and print their scores

In [ ]:
test_queries = {
    "Dog / puppy related": "A puppy is a friendly pet that enjoys walks.",
    "Dog / airplane unrelated": "The airplane carried passengers to a distant city.",
    "Travel related": "Tourists travel to beautiful cities and book hotels.",
    "Technology related": "A computer needs a fast processor and enough memory.",
    "Pets related": "Friendly pets enjoy walks, homes, and gentle care."
}

for test_name, query in test_queries.items():
    print("=" * 80)
    print(test_name)
    print("Query:", query)
    print("=" * 80)

    for rank, (sentence, score) in enumerate(find_similar(query, corpus, top_k=3), start=1):
        print(f"{rank}. {score:.3f} -> {sentence}")
    print()

### How should the scores be interpreted?

- **Closer to 1:** vectors have a similar direction and strong weighted-term overlap.
- **Closer to 0:** vectors have little or no shared weighted vocabulary.
- A high score is **not automatically proof of identical meaning**.
- A low score does not necessarily mean two sentences are unrelated in human language.

TF-IDF is a strong, interpretable baseline, but semantic embeddings are usually better when true semantic similarity is required.

## 12. Visualise the 10×10 matrix as a heatmap

A heatmap lets us see the clustering pattern instead of reading 100 numbers.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

heatmap = ax.imshow(similarity_matrix, cmap="viridis", vmin=0, vmax=1)

ax.set_title("10×10 TF-IDF Cosine Similarity Matrix")
ax.set_xlabel("Sentences")
ax.set_ylabel("Sentences")

ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)

for i in range(len(corpus)):
    for j in range(len(corpus)):
        ax.text(
            j, i,
            f"{similarity_matrix[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )

fig.colorbar(heatmap, ax=ax, label="Cosine similarity")
plt.tight_layout()
plt.show()

### New matplotlib commands explained

- `plt.subplots()` creates the plotting area.
- `ax.imshow()` displays the matrix as an image/heatmap.
- `cmap="viridis"` selects the colour map.
- `vmin=0, vmax=1` keeps the scale consistent with cosine similarity.
- `set_xticks()` / `set_yticks()` define tick positions.
- `set_xticklabels()` / `set_yticklabels()` add meaningful sentence labels.
- `ax.text()` writes each numerical score inside its cell.
- `fig.colorbar()` adds the colour legend.
- `plt.tight_layout()` reduces label overlap.
- `plt.show()` displays the final plot.

## 13. Why angle matters more than raw distance

Imagine every sentence vector as an arrow from the origin. Two arrows may have different lengths because one sentence contains more information, but they can still point in a similar direction.

Cosine similarity normalises the vector lengths:

$$
cos(θ) = \frac{A · B}{||A|| ||B||}
$$

This matters for text because document length can change vector magnitude. A long document may have larger numerical values simply because it contains more terms. Cosine similarity focuses more on the **orientation/pattern of term weights** than on absolute length.

If we used Euclidean distance alone, document length could have a stronger influence. Cosine similarity is therefore a useful baseline for comparing TF-IDF text vectors.

### Why this matters for the project

The goal is not simply to ask whether two vectors are numerically close in space. We want to know whether their **patterns of important terms point in a similar direction**.

For TF-IDF text comparison, that is often more useful than raw distance.

However, cosine similarity does not magically create semantic understanding. If two sentences use completely different words for the same idea, TF-IDF may give them a low score. This is one reason modern NLP often uses embedding models.

## 14. Final project summary

The complete pipeline is:

**10 raw sentences**
↓
**TF-IDF vectorisation**
↓
**Numerical vectors**
↓
**Cosine similarity**
↓
**10×10 NumPy similarity matrix**
↓
**`find_similar()` search function**
↓
**Heatmap visualisation**

The main lesson is that **vectorisation makes mathematical comparison possible**. TF-IDF provides an interpretable representation based on term importance, and cosine similarity compares the direction of those vectors.

### Submission checklist

- [ ] Explain why plain text cannot be directly compared mathematically.
- [ ] Explain why numerical vectors are needed for machine processing.
- [ ] Use exactly 10 sentences covering three topics.
- [ ] Convert them using `TfidfVectorizer`.
- [ ] Calculate all pairwise cosine similarities.
- [ ] Store the result in a NumPy matrix.
- [ ] Implement `find_similar(query, corpus, top_k)`.
- [ ] Test dog/puppy and dog/airplane-style cases.
- [ ] Print multiple similarity scores.
- [ ] Create a labelled 10×10 matplotlib heatmap.
- [ ] Explain the cosine formula.
- [ ] Explain why cosine compares angle/direction rather than raw distance.
- [ ] Explain why that distinction is useful for text comparison.